In [ ]:
from google.cloud import bigquery
import warnings
import pandas as pd
from datasets import Dataset, DatasetInfo
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

client = bigquery.Client(project="ads-599-final")

In [ ]:
# Radiology Reports — ED Visit + Hospital Ward Window (action records + state context) ---
# Imaging studies ordered during the ED stay and subsequent hospital ward stay (pre-ICU).
# hadm_id carried through from the radiology table — NULL for ED-only patients, populated for admitted.
# Filter note_type = 'RR' for primary reports only (exclude 'AR' addenda).
# exam_name and cpt_code pulled from radiology_detail to identify imaging modality.
#
# Window logic:
#   - ED-only patients (hadm_id IS NULL):  ed_intime → ed_outtime
#   - Admitted patients (hadm_id IS NOT NULL): ed_intime → first_icu_intime (or dischtime if no ICU)

query_radiology = """
WITH cohort_subjects AS (
  SELECT
    ed_stay_id,
    subject_id,
    hadm_id,
    ed_intime,
    CASE
      WHEN hadm_id IS NOT NULL AND first_icu_intime IS NOT NULL
        THEN first_icu_intime
      WHEN hadm_id IS NOT NULL
        THEN dischtime
      ELSE ed_outtime
    END AS window_end
  FROM `ads-599-final.rl_project.cohort_base`
),
radiology_with_detail AS (
  SELECT
    r.note_id,
    r.subject_id,
    r.hadm_id,
    r.charttime,
    r.text AS report_text,
    r.note_type,
    MAX(CASE WHEN rd.field_name = 'exam_name' THEN rd.field_value END) AS exam_name,
    MAX(CASE WHEN rd.field_name = 'cpt_code'  THEN rd.field_value END) AS cpt_code
  FROM `physionet-data.mimiciv_note.radiology` r
  LEFT JOIN `physionet-data.mimiciv_note.radiology_detail` rd
    ON r.note_id = rd.note_id
    AND rd.field_name IN ('exam_name', 'cpt_code')
  WHERE r.note_type = 'RR'
  GROUP BY r.note_id, r.subject_id, r.hadm_id, r.charttime, r.text, r.note_type
)
SELECT
  cs.ed_stay_id,
  rwd.subject_id,
  rwd.hadm_id,
  rwd.note_id,
  rwd.charttime,
  rwd.exam_name,
  rwd.cpt_code,
  rwd.report_text,
  END AS in_radiology_coverage_window
FROM radiology_with_detail rwd
INNER JOIN cohort_subjects cs
  ON rwd.subject_id = cs.subject_id
  AND CAST(rwd.charttime AS DATETIME) >= cs.ed_intime
  AND CAST(rwd.charttime AS DATETIME) <= cs.window_end
ORDER BY cs.ed_stay_id, rwd.charttime
"""

print("Running Query 7: Radiology reports...")
df_radiology = client.query(query_radiology).to_dataframe()
print(f"Shape: {df_radiology.shape}")

print(f"Unique ED visits with imaging: {df_radiology['ed_stay_id'].nunique():,}")
print(f"Records with hadm_id (admitted patients): {df_radiology['hadm_id'].notna().sum():,}")
print(f"Records without hadm_id (ED-only patients): {df_radiology['hadm_id'].isna().sum():,}")
print(f"\nRadiology coverage window distribution:")
print(df_radiology['in_radiology_coverage_window'].value_counts())
print(f"\nTop exam types:\n{df_radiology['exam_name'].value_counts().head(10)}")
df_radiology.head()

Running Query 7: Radiology reports...
Shape: (576542, 9)
Unique ED visits with imaging: 269,928
Records with hadm_id (admitted patients): 379,806
Records without hadm_id (ED-only patients): 196,736

Radiology coverage window distribution:
in_radiology_coverage_window
False    576542
Name: count, dtype: Int64

Top exam types:
exam_name
CHEST (PA & LAT)                          114753
CHEST (PORTABLE AP)                        58672
CT HEAD W/O CONTRAST                       51425
CT ABD & PELVIS WITH CONTRAST              30151
CT C-SPINE W/O CONTRAST                    18768
LIVER OR GALLBLADDER US (SINGLE ORGAN)     16873
CT CHEST W/CONTRAST                         8846
CT ABD & PELVIS W/O CONTRAST                8491
ABDOMEN (SUPINE & ERECT)                    7726
CTA CHEST                                   7552
Name: count, dtype: int64


,ed_stay_id,subject_id,hadm_id,note_id,charttime,exam_name,cpt_code,report_text,in_radiology_coverage_window
0,30000012,11714491,<NA>,11714491-RR-29,2126-02-14 23:12:00,CHEST (PA & LAT),NaN,INDICATION: ___ woman with altered mental sta...,False
1,30000012,11714491,21562392,11714491-RR-30,2126-02-15 09:11:00,LIVER OR GALLBLADDER US (SINGLE ORGAN),NaN,EXAMINATION: LIVER OR GALLBLADDER US (SINGLE ...,False
2,30000012,11714491,21562392,11714491-RR-31,2126-02-16 17:16:00,CT CHEST W/O CONTRAST,NaN,EXAMINATION: CT chest noncontrast.\n\nINDICAT...,False
3,30000012,11714491,21562392,11714491-RR-32,2126-02-17 00:38:00,MRI LIVER W&W/O CONTRAST,NaN,EXAMINATION: MRI of the Abdomen\n\nINDICATION...,False
4,30000038,13821532,26255538,13821532-RR-28,2152-12-08 10:41:00,CHEST (PA & LAT),NaN,EXAMINATION: CHEST (PA AND LAT)\n\nINDICATION...,False


In [ ]:
PHYSIONET_LICENSE = "PhysioNet Credentialed Health Data License 1.5.0 — https://physionet.org/content/mimiciv/view-license/3.1/"


info = DatasetInfo(
    description=(
        "Radiology report text from mimiciv_note.radiology for cohort patients. "
        "Covers all imaging modalities (CXR, CT, MRI, ultrasound, etc.). Primary reports only (note_type=RR). "
        "Window covers ED arrival through hospital ward stay (capped at ICU transfer if applicable). "
        "hadm_id is NULL for ED-only patients and populated for admitted patients. "
        "exam_name and cpt_code included from radiology_detail to identify imaging modality."
    ),
    license=PHYSIONET_LICENSE,
)

ds = Dataset.from_pandas(df_radiology, split="radiology_details", info=radiology_info)
ds.push_to_hub("ADS599-Capstone/raw_data", config_name="radiology_details", data_dir='radiology')
print("Pushed radiology data to HuggingFace Hub.")